## **Building a Convolutional Neural Network (CNN) Architecture in PyTorch** 
___

This task translates a CNN architecture diagram into a working PyTorch `nn.Module` class. The diagram specifies a deep convolutional network: an input image passes through a convolution + pooling block, then three consecutive convolutions and a second pooling block, followed by dropout, flattening, and three fully connected layers ending in a softmax classifier.

The core challenge is calculating the spatial dimensions (height/width) at each stage, since the diagram left several of these blank (?) for the student to work out using the convolution/pooling output-size formula:

<div class="alert alert-block alert-success" style="font-family: Times New Roman">
    <h4><strong>Laboratory Activity 6</strong></h4>

<p style="font-family:Times New Roman; text-align:justify; font-size:15px">
    <b>Instruction:</b> Convert the following CNN architecture diagram into a PyTorch CNN Architecture.
</p>

<center><img src="figures-quick_draw.png" width="400px"></center>
</div>

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random

In [6]:
# ---------- Seed function----------
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)

# ---------- Shape calculator ----------
def calc_out(w, f, s, p):
    print(f'Output Shape: {(w - f + 2 * p) // s + 1}')

set_seed(42)

# ---------- Dummy input to trace shapes (batch=32) ----------
images = torch.randn(32, 1, 28, 28)
print("Input shape:", images.shape)

Input shape: torch.Size([32, 1, 28, 28])


In [7]:
# Conv1 must output 28x28
calc_out(28, 3, 1, 1)

conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=(3,3), stride=1, padding=1)

x = F.relu(conv1(images))
print(x.shape)

Output Shape: 28
torch.Size([32, 32, 28, 28])


In [8]:
# Pool1 must output 15x15 (note: padding=1 here, unlike a typical pool)
calc_out(28, 2, 2, 1)

pool1 = nn.MaxPool2d(kernel_size=(2,2), stride=2, padding=1)

x = pool1(x)
print(x.shape)

Output Shape: 15
torch.Size([32, 32, 15, 15])


In [9]:
# Each of these keeps spatial size the same (kernel=3, stride=1, padding=1 = "same" padding)
calc_out(15, 3, 1, 1)

conv2 = nn.Conv2d(32, 64, (3,3), 1, 1)
conv3 = nn.Conv2d(64, 128, (3,3), 1, 1)
conv4 = nn.Conv2d(128, 256, (3,3), 1, 1)

x = F.relu(conv2(x))
print(x.shape)

x = F.relu(conv3(x))
print(x.shape)

x = F.relu(conv4(x))
print(x.shape)

Output Shape: 15
torch.Size([32, 64, 15, 15])
torch.Size([32, 128, 15, 15])
torch.Size([32, 256, 15, 15])


In [10]:
# Pool2 must output 7x7
calc_out(15, 2, 2, 0)

pool2 = nn.MaxPool2d((2,2), 2)

x = pool2(x)
print(x.shape)

Output Shape: 7
torch.Size([32, 256, 7, 7])


In [11]:
256 * 7 * 7   # each instance contains this many values after flattening

dropout = nn.Dropout(p=0.2)
x = dropout(x)

flat = x.view(x.size(0), -1)
print(flat.shape)

torch.Size([32, 12544])


In [12]:
fcn1 = nn.Linear(256*7*7, 1000)
fcn2 = nn.Linear(1000, 500)
fcn3 = nn.Linear(500, 10)   # replace 10 with your actual number of classes

out = F.relu(fcn1(flat))
out = F.relu(fcn2(out))
out = F.softmax(fcn3(out), dim=1)
print(out.shape)

torch.Size([32, 10])


In [13]:
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, (3,3), 1, 1)
        self.pool1 = nn.MaxPool2d((2,2), 2, padding=1)
        self.conv2 = nn.Conv2d(32, 64, (3,3), 1, 1)
        self.conv3 = nn.Conv2d(64, 128, (3,3), 1, 1)
        self.conv4 = nn.Conv2d(128, 256, (3,3), 1, 1)
        self.pool2 = nn.MaxPool2d((2,2), 2)
        self.dropout = nn.Dropout(p=0.2)
        self.fcn1 = nn.Linear(256*7*7, 1000)
        self.fcn2 = nn.Linear(1000, 500)
        self.fcn3 = nn.Linear(500, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)

        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool2(x)

        x = self.dropout(x)
        x = x.view(x.size(0), -1)

        x = F.relu(self.fcn1(x))
        x = F.relu(self.fcn2(x))
        x = F.softmax(self.fcn3(x), dim=1)
        return x

In [14]:
model = CNN(num_classes=10)   # replace 10 with your dataset's actual number of classes
model

CNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=(2, 2), stride=2, padding=1, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=(2, 2), stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.2, inplace=False)
  (fcn1): Linear(in_features=12544, out_features=1000, bias=True)
  (fcn2): Linear(in_features=1000, out_features=500, bias=True)
  (fcn3): Linear(in_features=500, out_features=10, bias=True)
)

In [15]:
def count_parameters(model):
    params = [p.numel() for p in model.parameters() if p.requires_grad]
    for item in params:
        print(f'{item:>8}')
    print(f'________\n{sum(params):>8}')

count_parameters(model)

     288
      32
   18432
      64
   73728
     128
  294912
     256
12544000
    1000
  500000
     500
    5000
      10
________
13438350


In [16]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(device)

cpu


In [17]:
model = CNN(num_classes=10)  # adjust num_classes to your dataset
dummy_input = torch.randn(32, 1, 28, 28)
output = model(dummy_input)
print(output.shape)  # should print torch.Size([32, num_classes])

torch.Size([32, 10])
